In [2]:
import pandas as pd
import numpy as np

from pathlib import Path

In [3]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

In [4]:
BASE_DIR = Path("..")

DATACO_RAW = (
    BASE_DIR/"datasets"/"raw"/"dataco"/"DataCoSupplyChainDataset.csv"
)

SMART_RAW=(
    BASE_DIR/"datasets"/"raw"/"smart_logistics" /"Smart_Logistics.csv"
)

PROCESSED_DIR=(
    BASE_DIR/"datasets"/"processed"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [5]:
##define column names
def clean_column_names(df):
    df = df.copy()

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("(", "", regex=False)
        .str.replace(")", "", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )

    return df

In [6]:
#remove white Space 

def strip_string_columns(df):
    df = df.copy()

    string_columns = df.select_dtypes(
        include="object"
    ).columns

    for column in string_columns:
        df[column] = df[column].str.strip()

    return df

In [7]:
# Convert empty strings to missing values
MISSING_TOKENS = [
    "",
    " ",
    "NA",
    "N/A",
    "na",
    "n/a",
    "NULL",
    "null",
    "None",
    "none"
]



In [8]:
# Build a missing-value report
def missing_report(df):
    report = pd.DataFrame({
        "missing_count": df.isnull().sum(),
        "missing_percentage": (
            df.isnull().sum()
            / len(df)
            * 100
        )
    })

    return report.sort_values(
        "missing_percentage",
        ascending=False
    )

In [ ]:
# Check near-constant columns
def near_constant_columns(
    df,
    threshold=0.99
):
    result = []

    for column in df.columns:
        top_frequency = (
            df[column]
            .value_counts(
                normalize=True,
                dropna=False
            )
            .iloc[0]
        )

        if top_frequency >= threshold:
            result.append(
                (column, top_frequency)
            )

    return result


In [94]:
# the final general validation of your cleaned datasets
def validate_dataframe(df, name):
    print(f"\n{'=' * 60}")
    print(f"{name} VALIDATION")
    print(f"{'=' * 60}")

    print("Rows:", len(df))
    print("Columns:", len(df.columns))

    print(
        "Duplicate rows:",
        df.duplicated().sum()
    )

    print(
        "Total missing values:",
        df.isnull().sum().sum()
    )

    print(
        "Duplicate column names:",
        df.columns.duplicated().sum()
    )

    print("\nData types:")
    print(df.dtypes.value_counts())

-----------------------------------------------------Data Co SupplyChain Dataset -------------------------------------------------------

In [10]:
dataco_raw = pd.read_csv(
    DATACO_RAW, encoding_errors="ignore"
)
print("dataco:", dataco_raw.shape)


dataco: (180519, 53)


In [11]:
dataco = dataco_raw.copy()

In [12]:
dataco = clean_column_names(dataco)

In [13]:
# dataco.columns.tolist()

In [14]:
##check duplicate step(9)
print(dataco.columns[dataco.columns.duplicated()].tolist())

[]


In [15]:
##remove white space 
dataco = strip_string_columns(dataco)

C:\Users\anujk\AppData\Local\Temp\ipykernel_18980\2917679433.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = df.select_dtypes(


In [16]:
# Convert empty strings to missing values
dataco = dataco.replace(
    MISSING_TOKENS,
    np.nan
)

In [ ]:
##Rechecking Missing Values
dataco.isnull().sum().sort_values(
    ascending= False
)

In [42]:
dataco_missing = missing_report(dataco)
dataco_missing

,missing_count,missing_percentage
product_description,180519,100.000000
order_zipcode,155679,86.239676
customer_lname,8,0.004432
customer_zipcode,3,0.001662
days_for_shipment_scheduled,0,0.000000
sales_per_customer,0,0.000000
benefit_per_order,0,0.000000
delivery_status,0,0.000000
late_delivery_risk,0,0.000000
customer_city,0,0.000000


In [47]:
dataco_missing[
    dataco_missing["missing_percentage"] > 50
]

,missing_count,missing_percentage
product_description,180519,100.000000
order_zipcode,155679,86.239676


In [48]:
[column for column in dataco.columns if "date" in column]


['order_date_dateorders', 'shipping_date_dateorders']

In [49]:
date_columns = [
    "order_date",
    "shipping_date"
]

for column in date_columns:
    if column in dataco.columns:
        dataco[column] = pd.to_datetime(
            dataco[column],
            errors="coerce"
        )

In [56]:
if "order_date" in dataco.columns:
    print("DataCo order date range:")
    print(dataco["order_date"].min())
    print(dataco["order_date"].max())

In [58]:
##check impossible date relationship 
if {
    "order_date",
    "shipping_date"
}.issubset(dataco.columns):
    invalid_dates = dataco[
        dataco["shipping_date"] < dataco["order_date"]
    ]

    print("invalid shipping dates", len(invalid_dates))

In [59]:
# Check numerical columns
dataco.select_dtypes(
    include= np.number
).describe().T

,count,mean,std,min,25%,50%,75%,max
days_for_shipping_real,180519.0,3.497654,1.623722,0.000000,2.000000,3.000000,5.000000,6.000000
days_for_shipment_scheduled,180519.0,2.931847,1.374449,0.000000,2.000000,4.000000,4.000000,4.000000
benefit_per_order,180519.0,21.974989,104.433526,-4274.979980,7.000000,31.520000,64.800003,911.799988
sales_per_customer,180519.0,183.107609,120.043670,7.490000,104.379997,163.990005,247.399994,1939.989990
late_delivery_risk,180519.0,0.548291,0.497664,0.000000,0.000000,1.000000,1.000000,1.000000
category_id,180519.0,31.851451,15.640064,2.000000,18.000000,29.000000,45.000000,76.000000
customer_id,180519.0,6691.379495,4162.918106,1.000000,3258.500000,6457.000000,9779.000000,20757.000000
customer_zipcode,180516.0,35921.126914,37542.461122,603.000000,725.000000,19380.000000,78207.000000,99205.000000
department_id,180519.0,5.443460,1.629246,2.000000,4.000000,5.000000,7.000000,12.000000
latitude,180519.0,29.719955,9.813646,-33.937553,18.265432,33.144863,39.279617,48.781933


In [17]:
shipping_columns = [
    "days_for_shipping_real",
    "days_for_shipment_scheduled"
]

existing_shipping_columns = [
    column
    for column in shipping_columns
    if column in dataco.columns
]

dataco[existing_shipping_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
days_for_shipping_real,180519.0,3.497654,1.623722,0.0,2.0,3.0,5.0,6.0
days_for_shipment_scheduled,180519.0,2.931847,1.374449,0.0,2.0,4.0,4.0,4.0


In [18]:
for column in existing_shipping_columns:
    print(
        column,
        "negative values:",
        (dataco[column] < 0).sum()
    )

days_for_shipping_real negative values: 0
days_for_shipment_scheduled negative values: 0


In [61]:
##Validate DataCO Target
if "late_delivery_risk" in dataco.columns:
    print(
        dataco["late_delivery_risk"].value_counts(dropna=False)
    )

late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64


In [ ]:
# if "late_delivery_risk" in dataco.columns:
#     print(
#         "Unexpected values:",
#         sorted(
#             set(
#                 dataco["late_delivery_risk"]
#                 .dropna()
#                 .unique()
#             )
#             - {0, 1}
#         )
#     )

Unexpected values: []


In [63]:
# Duplicate analysis
dataco_duplicate_count = dataco.duplicated().sum()

print(dataco_duplicate_count)

0


In [65]:
##remove unwanted columns
senstive_columns=[
    "customer_password",
    "customer_email",
    "customer_fname",
    "customer_lname",
    "customer_street",
    "product_image",
    "product_description",
    "order_zipcode"
]



In [68]:
columns_to_remove = [
    column
    for column in senstive_columns
    if column in dataco.columns
]

print(columns_to_remove)

['customer_password', 'customer_email', 'customer_fname', 'customer_lname', 'customer_street', 'product_image', 'product_description', 'order_zipcode']


In [69]:
dataco = dataco.drop(
    columns = columns_to_remove
)

In [71]:
# dataco.columns

check_dataco_columns=[
    column
    for column in dataco.columns
    if dataco[column].nunique(dropna=False) <= 1
]

print("dataco constant columns", check_dataco_columns)

dataco constant columns ['product_status']


In [75]:
near_constant_columns(dataco)

[('product_status', np.float64(1.0))]

In [ ]:
# Delete product_status columns (because it has constant value)
# dataco = dataco.drop(columns=["product_status"])
# print("product_status" in dataco.columns)


In [95]:
validate_dataframe(
    dataco,
    "DataCo"
)


DataCo VALIDATION
Rows: 180519
Columns: 44
Duplicate rows: 0
Total missing values: 3
Duplicate column names: 0

Data types:
str        18
int64      13
float64    13
Name: count, dtype: int64


In [78]:
dataco.info()

<class 'pandas.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 45 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   type                         180519 non-null  str    
 1   days_for_shipping_real       180519 non-null  int64  
 2   days_for_shipment_scheduled  180519 non-null  int64  
 3   benefit_per_order            180519 non-null  float64
 4   sales_per_customer           180519 non-null  float64
 5   delivery_status              180519 non-null  str    
 6   late_delivery_risk           180519 non-null  int64  
 7   category_id                  180519 non-null  int64  
 8   category_name                180519 non-null  str    
 9   customer_city                180519 non-null  str    
 10  customer_country             180519 non-null  str    
 11  customer_id                  180519 non-null  int64  
 12  customer_segment             180519 non-null  str    
 13  customer_s

In [79]:
missing_report(dataco)

,missing_count,missing_percentage
customer_zipcode,3,0.001662
days_for_shipping_real,0,0.000000
days_for_shipment_scheduled,0,0.000000
benefit_per_order,0,0.000000
type,0,0.000000
delivery_status,0,0.000000
late_delivery_risk,0,0.000000
category_id,0,0.000000
category_name,0,0.000000
customer_city,0,0.000000


In [83]:
dataco_raw.shape

(180519, 53)

In [84]:
print("Raw DataCo:", dataco_raw.shape)
print("Clean DataCo:", dataco.shape)

Raw DataCo: (180519, 53)
Clean DataCo: (180519, 44)


In [89]:
cleaning_summary_dataco={
    "dataco_raw_rows": len(dataco_raw),
    "dataco_clean_rows": len(dataco),
    "dataco_rows_removed": (
        len(dataco_raw) - len(dataco)
    )
}
cleaning_summary_dataco

{'dataco_raw_rows': 180519,
 'dataco_clean_rows': 180519,
 'dataco_rows_removed': 0}

In [90]:
expected_dataco_columns = [
    "order_id",
    "order_status",
    "shipping_mode",
    "delivery_status",
    "late_delivery_risk",
    "days_for_shipping_real",
    "days_for_shipment_scheduled"
]

missing_expected_dataco = [
    column
    for column in expected_dataco_columns
    if column not in dataco.columns
]

print(
    "Missing expected DataCo columns:",
    missing_expected_dataco
)

Missing expected DataCo columns: []


In [99]:
dataco_output = (
    PROCESSED_DIR
    / "dataco_cleaned.csv"
)

dataco.to_csv(
    dataco_output,
    index=False
)

In [100]:
print(
    "Saved:",
    dataco_output
)

Saved: ..\datasets\processed\dataco_cleaned.csv


In [102]:
dataco_check = pd.read_csv(
    dataco_output
)

In [105]:
print(
    "DataCo saved shape:",
    dataco_check.shape
)

DataCo saved shape: (180519, 44)


In [104]:
print(
    "DataCo duplicates:",
    dataco_check.duplicated().sum()
)

DataCo duplicates: 0


-----------------------------------------------This part is For Smart CO Dataset-----------------------------------------------------  

In [46]:
smart_raw = pd.read_csv(
    SMART_RAW, encoding_errors="ignore"
)

print("Smart Co: ", smart_raw.shape)

Smart Co:  (1000, 16)


In [47]:
smart = smart_raw.copy()

In [48]:
smart = clean_column_names(smart)

In [49]:
# smart.columns.tolist()

In [50]:
##check duplicate 
print(smart.columns[smart.columns.duplicated()].tolist())

[]


In [51]:
##remove white space
smart = strip_string_columns(smart)



C:\Users\anujk\AppData\Local\Temp\ipykernel_18980\2917679433.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = df.select_dtypes(


In [52]:
# Convert empty strings to missing values
smart = smart.replace(
    MISSING_TOKENS,
    np.nan
)

In [53]:
smart.isnull().sum().sort_values(
    ascending=True
)



timestamp                    0
asset_id                     0
latitude                     0
longitude                    0
inventory_level              0
shipment_status              0
temperature                  0
humidity                     0
traffic_status               0
waiting_time                 0
user_transaction_amount      0
user_purchase_frequency      0
asset_utilization            0
demand_forecast              0
logistics_delay              0
logistics_delay_reason     263
dtype: int64

In [54]:
smart_missing = missing_report(smart)
smart_missing

,missing_count,missing_percentage
logistics_delay_reason,263,26.3
timestamp,0,0.0
latitude,0,0.0
asset_id,0,0.0
inventory_level,0,0.0
shipment_status,0,0.0
temperature,0,0.0
longitude,0,0.0
humidity,0,0.0
traffic_status,0,0.0


In [ ]:
# smart_missing [
#     smart_missing["missing_percentage"] > 50
# ]

,missing_count,missing_percentage


In [55]:
##smart logistics timestamp
[column for column in smart.columns if "time" in column or "date" in column]

['timestamp', 'waiting_time']

In [28]:
if "timestamp" in smart.columns:
    smart["timestamp"] = pd.to_datetime(
        smart["timestamp"],
        errors="coerce"
    )

In [29]:
smart["timestamp"].isnull().sum()

np.int64(0)

In [30]:
##date Range

if "timestamp" in smart.columns:
    print("smart logistics timestamp range: ")
    print(smart["timestamp"].min())
    print(smart["timestamp"].max())

smart logistics timestamp range: 
2024-01-01 11:37:57
2024-12-30 20:21:58


In [31]:
##check numerical columns
smart.select_dtypes(
    include=np.number
).describe().T

,count,mean,std,min,25%,50%,75%,max
latitude,1000.0,-1.360093,51.997183,-89.7915,-46.167975,-4.50315,44.50280,89.8701
longitude,1000.0,0.837049,104.843618,-179.8202,-88.448075,0.67830,88.15645,179.9237
inventory_level,1000.0,297.915000,113.554773,100.0000,201.000000,299.00000,399.00000,500.0000
temperature,1000.0,23.893900,3.322178,18.0000,21.200000,23.80000,26.60000,30.0000
humidity,1000.0,65.042200,8.753765,50.0000,57.200000,65.20000,72.40000,80.0000
waiting_time,1000.0,35.062000,14.477768,10.0000,23.000000,35.00000,49.00000,60.0000
user_transaction_amount,1000.0,299.055000,117.787792,100.0000,191.750000,301.50000,405.00000,500.0000
user_purchase_frequency,1000.0,5.513000,2.935379,1.0000,3.000000,6.00000,8.00000,10.0000
asset_utilization,1000.0,79.599100,11.631153,60.0000,69.475000,79.25000,89.42500,100.0000
demand_forecast,1000.0,199.284000,59.920847,100.0000,144.000000,202.00000,251.25000,300.0000


In [ ]:
# Validate Smart Logistics temperature
if "temperature" in smart.columns:
    print(smart["temperature"].describe())

count    1000.000000
mean       23.893900
std         3.322178
min        18.000000
25%        21.200000
50%        23.800000
75%        26.600000
max        30.000000
Name: temperature, dtype: float64


In [33]:
if "temperature" in smart.columns:
    suspicious_temperature = smart[
        (smart["temperature"] < -50)
        |
        (smart["temperature"] > 70)
    ]

    print(
        "Suspicious temperature records:",
        len(suspicious_temperature)
    )

Suspicious temperature records: 0


In [34]:
#Validate Humidity

if "humidity" in smart.columns:
    print("Humidity below 0: ", (smart["humidity"] < 0).sum())
    print("Humidity above 100", (smart["humidity"] > 100).sum())

Humidity below 0:  0
Humidity above 100 0


In [35]:
# Validate asset utilization
if "asset_utilization" in smart.columns:
    print(
        smart["asset_utilization"].describe()
    )

count    1000.000000
mean       79.599100
std        11.631153
min        60.000000
25%        69.475000
50%        79.250000
75%        89.425000
max       100.000000
Name: asset_utilization, dtype: float64


In [36]:
# Validate GPS
if {
    "latitude",
    "longitude"
}.issubset(smart.columns):

    print(
        smart[
            ["latitude", "longitude"]
        ].describe()
    )

          latitude    longitude
count  1000.000000  1000.000000
mean     -1.360093     0.837049
std      51.997183   104.843618
min     -89.791500  -179.820200
25%     -46.167975   -88.448075
50%      -4.503150     0.678300
75%      44.502800    88.156450
max      89.870100   179.923700


In [37]:
if "latitude" in smart.columns:
    print(
        "Invalid latitude:",
        (
            (smart["latitude"] < -90)
            |
            (smart["latitude"] > 90)
        ).sum()
    )

Invalid latitude: 0


In [38]:
if "longitude" in smart.columns:
    print(
        "Invalid longitude:",
        (
            (smart["longitude"] < -180)
            |
            (smart["longitude"] > 180)
        ).sum()
    )

Invalid longitude: 0


In [ ]:
##traffic analysis
smart["traffic_status"].value_counts(dropna=False)

traffic_status
Detour    345
Clear     328
Heavy     327
Name: count, dtype: int64

In [57]:
smart["traffic_status"] = (
    smart["traffic_status"]
    .str.strip()
    .str.title()
)

In [58]:
smart["traffic_status"].value_counts(dropna=False)

traffic_status
Detour    345
Clear     328
Heavy     327
Name: count, dtype: int64

In [ ]:
#validate Traffic Status

if "traffic_status" in smart.columns:
    print(
        smart["traffic_status"]
        .value_counts(dropna=False)
    )

traffic_status
Detour    345
Clear     328
Heavy     327
Name: count, dtype: int64


In [45]:
smart["traffic_status"].unique()

<StringArray>
[nan]
Length: 1, dtype: str

In [40]:
if "shipment_status" in smart.columns:
    print(
        smart["shipment_status"]
        .value_counts(dropna=False)
    )

shipment_status
Delayed       350
Delivered     338
In Transit    312
Name: count, dtype: int64


In [60]:
# Validate target values
if "logistics_delay" in smart.columns:
    print(
        smart["logistics_delay"].value_counts(dropna=False)
    )

logistics_delay
1    566
0    434
Name: count, dtype: int64


In [64]:
# Duplicate analysis
smart_duplicate_count = smart.duplicated().sum()
print(
    "Smart Logistics duplicates:",
    smart_duplicate_count
)

Smart Logistics duplicates: 0


In [73]:
smart_constant_columns=[
    column
    for column in smart.columns
    if smart[column].nunique(dropna=False) <= 1
]

print("\nSmart Logistics constant columns:")
print(smart_constant_columns)


Smart Logistics constant columns:
[]


In [76]:
near_constant_columns(smart)

[]

In [77]:
smart.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   timestamp                1000 non-null   str    
 1   asset_id                 1000 non-null   str    
 2   latitude                 1000 non-null   float64
 3   longitude                1000 non-null   float64
 4   inventory_level          1000 non-null   int64  
 5   shipment_status          1000 non-null   str    
 6   temperature              1000 non-null   float64
 7   humidity                 1000 non-null   float64
 8   traffic_status           1000 non-null   str    
 9   waiting_time             1000 non-null   int64  
 10  user_transaction_amount  1000 non-null   int64  
 11  user_purchase_frequency  1000 non-null   int64  
 12  logistics_delay_reason   737 non-null    str    
 13  asset_utilization        1000 non-null   float64
 14  demand_forecast          1000 non-nu

In [80]:
missing_report(smart)

,missing_count,missing_percentage
logistics_delay_reason,263,26.3
timestamp,0,0.0
latitude,0,0.0
asset_id,0,0.0
inventory_level,0,0.0
shipment_status,0,0.0
temperature,0,0.0
longitude,0,0.0
humidity,0,0.0
traffic_status,0,0.0


In [ ]:
smart.shape

(1000, 16)

In [86]:
print("Raw smart: ", smart_raw.shape)
print("cleaned smart: ", smart.shape)

Raw smart:  (1000, 16)
cleaned smart:  (1000, 16)


In [87]:
cleaning_summary_smart={
    "smart_raw_rows": len(smart_raw),
    "smart_clean_rows": len(smart),
    "smart_rows_removed": (
        len(smart_raw) - len(smart)
    )
}
cleaning_summary_smart

{'smart_raw_rows': 1000, 'smart_clean_rows': 1000, 'smart_rows_removed': 0}

In [91]:
expected_smart_columns = [
    "timestamp",
    "asset_id",
    "latitude",
    "longitude",
    "shipment_status",
    "temperature",
    "humidity",
    "traffic_status",
    "waiting_time",
    "logistics_delay_reason",
    "asset_utilization",
    "logistics_delay"
]

missing_expected_smart = [
    column
    for column in expected_smart_columns
    if column not in smart.columns
]

print(
    "Missing expected Smart Logistics columns:",
    missing_expected_smart
)

Missing expected Smart Logistics columns: []


In [97]:
pd.crosstab(
    smart["logistics_delay"],
    smart["logistics_delay_reason"],
    dropna=False
)

logistics_delay_reason,Mechanical Failure,Traffic,Weather,NaN
logistics_delay,,,,
0,101,101,116,116
1,133,135,151,147


In [98]:
print(smart.columns.tolist())

['timestamp', 'asset_id', 'latitude', 'longitude', 'inventory_level', 'shipment_status', 'temperature', 'humidity', 'traffic_status', 'waiting_time', 'user_transaction_amount', 'user_purchase_frequency', 'logistics_delay_reason', 'asset_utilization', 'demand_forecast', 'logistics_delay']


In [96]:
validate_dataframe(
    smart,
    "Smart Logistics"
)


Smart Logistics VALIDATION
Rows: 1000
Columns: 16
Duplicate rows: 0
Total missing values: 263
Duplicate column names: 0

Data types:
int64      6
str        5
float64    5
Name: count, dtype: int64


In [101]:
smart_output = (
    PROCESSED_DIR
    / "smart_logistics_cleaned.csv"
)

smart.to_csv(
    smart_output,
    index=False
)

In [106]:

smart_check = pd.read_csv(
    smart_output
)

In [107]:
print(
    "Smart Logistics saved shape:",
    smart_check.shape
)

Smart Logistics saved shape: (1000, 16)


In [108]:
print(
    "Smart duplicates:",
    smart_check.duplicated().sum()
)

Smart duplicates: 0
